In [1]:
import duckdb
from pathlib import Path
import gc

In [3]:
# 1. Identify the current directory (Handles both .py scripts and Jupyter)
try:
    # Use __file__ for standalone scripts
    current_location = Path(__file__).resolve().parent
except NameError:
    # Use Current Working Directory for Jupyter/Interactive sessions
    current_location = Path.cwd().resolve()

# 2. Navigate to the actual Project Root
# If current_location is 'Finalized_Scripts', move to the parent directory
if current_location.name in ["Finalized_Scripts", "Test_Scripts", "scripts"]:
    project_root = current_location.parent
else:
    project_root = current_location

# 3. Define and Verify IO Paths
DATA_DIR = project_root / "Data_Files"
INPUT_PATTERN = project_root / "Data_Files"/"floodnet_parquet_data" / "*.parquet"

# Ensure output directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 4. Output Filenames
MERGED_OUT = DATA_DIR / "floodnet_full_dataset_merged.parquet"
FLOODS_OUT = DATA_DIR / "floodnet_floods_only.parquet"

In [5]:
# 4. Initialize DuckDB Connection
con = duckdb.connect()

print(f"Beginning merge of sensor data from: {INPUT_PATTERN.parent}")

# Step A: Merge all sensors into the master dataset
# We convert Path objects to strings for the SQL execution
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{str(INPUT_PATTERN)}', union_by_name=True)
    ) 
    TO '{str(MERGED_OUT)}' (FORMAT PARQUET, COMPRESSION 'ZSTD');
""")
print(f"Full dataset merged successfully to: {MERGED_OUT}")

# Step B: Isolate flood events (>10mm processed depth)
print("Extracting flood events (>10mm processed depth)...")
con.execute(f"""
    COPY (
        SELECT * FROM read_parquet('{str(INPUT_PATTERN)}', union_by_name=True)
        WHERE depth_proc_mm > 10
    ) 
    TO '{str(FLOODS_OUT)}' (FORMAT PARQUET, COMPRESSION 'ZSTD');
""")
print(f"Flood events isolated and saved to: {FLOODS_OUT}")

# 5. Data Verification
total_rows = con.execute(f"SELECT count(*) FROM '{str(MERGED_OUT)}'").fetchone()[0]
flood_rows = con.execute(f"SELECT count(*) FROM '{str(FLOODS_OUT)}'").fetchone()[0]

print(f"\nETL Summary:")
print(f"Total rows processed: {total_rows:,}")
print(f"Total flood events isolated: {flood_rows:,}")

# Explicitly close connection and clear memory
con.close()
gc.collect()

Beginning merge of sensor data from: /sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/Data_Files/floodnet_parquet_data


RuntimeError: Query interrupted